In [2]:
# Imports needed
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

import random, math

seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)


In [3]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/ECG_10s_2500.csv")

print("Data shape (raw):", df.shape)

# First 2500 columns are ECG samples we care about
signal_cols = df.columns[:2500]
X = df[signal_cols].values.astype(np.float32)
# We also care about label, won't use paitient ID
y = df['label'].astype(int).values

# Per-segment z-normalization, keep normalized scale for GAN and classifier
X_mean = X.mean(axis=1, keepdims=True)
X_std  = X.std(axis=1, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

timesteps = X.shape[1]
X = X.reshape((-1, timesteps, 1))

print("X shape:", X.shape)
print("Label distribution:", np.bincount(y))


Mounted at /content/drive
Data shape (raw): (45630, 2505)
X shape: (45630, 2500, 1)
Label distribution: [43169  2461]


In [4]:
# Startified split to ensure enough hypoglycemic signals are in test, train and validation
# A split of 70/15/15 was used
indices = np.arange(len(X))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, stratify=y, random_state=seed
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=seed+1
)

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

print("Train:", X_train.shape, "Pos:", y_train.sum())
print("Val:  ", X_val.shape,   "Pos:", y_val.sum())
print("Test: ", X_test.shape,  "Pos:", y_test.sum())
print("Train pos ratio:", y_train.mean())


Train: (31941, 2500, 1) Pos: 1723
Val:   (6844, 2500, 1) Pos: 369
Test:  (6845, 2500, 1) Pos: 369
Train pos ratio: 0.05394320778936164


In [5]:
# Training the GAN Model to generate synthetic data to increase number of hypoglycemic signals
# Extract positive training samples for GAN
X_pos = X_train[y_train == 1]
n_pos = len(X_pos)
print("GAN training set (positives only):", X_pos.shape)

if n_pos == 0:
    print("No positive samples → GAN will be skipped.")
    # Define dummies so later cells don't crash
    latent_dim = 100
    generator = tf.keras.Sequential(name="dummy_generator")
    discriminator = tf.keras.Sequential(name="dummy_discriminator")
else:
# Hyperparameter:
    timesteps = X_pos.shape[1]
    latent_dim = 128
    batch_size = 64
    steps = 6000
    print_interval = 200

    # ratio between Discriminator and Generator updates
    d_updates_per_step = 1
    g_updates_per_step = 1


    def make_generator(latent_dim=128, out_len=2500):
# Here we define the generator architecture
        base_len = out_len // 4

        model = tf.keras.Sequential(name="generator_1d_ecg")
        model.add(layers.Input(shape=(latent_dim,)))
        model.add(layers.Dense(base_len * 64))
        model.add(layers.LeakyReLU(0.2)) # leaky relu to avoid the idea that normal relu might lead to dead neurons (setting negatives to a 0)
        model.add(layers.Reshape((base_len, 64)))

        model.add(layers.UpSampling1D(size=2))
        model.add(layers.Conv1D(64, kernel_size=7, padding="same"))
        model.add(layers.LeakyReLU(0.2))
        model.add(layers.BatchNormalization())

        model.add(layers.UpSampling1D(size=2))
        model.add(layers.Conv1D(32, kernel_size=7, padding="same"))
        model.add(layers.LeakyReLU(0.2))
        model.add(layers.BatchNormalization())

        # Final conv to single channel, linear activation
        model.add(layers.Conv1D(1, kernel_size=7, padding="same", activation="linear"))
# Final output of generator is linear so it provided an ECG signal
        return model

    def make_discriminator(input_len=2500):
# Here we define the discriminator architecture
        inp = layers.Input(shape=(input_len, 1))

        x = layers.Conv1D(32, kernel_size=7, strides=2, padding="same")(inp)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.25)(x) # added dropout to avoid overfitting

        x = layers.Conv1D(64, kernel_size=7, strides=2, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.25)(x)

        x = layers.Conv1D(128, kernel_size=7, strides=2, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.25)(x)

        x = layers.Flatten()(x)
        x = layers.Dense(128)(x)
        x = layers.LeakyReLU(0.2)(x)
        out = layers.Dense(1, activation="sigmoid")(x)
# Output of discriminator is a sigmoid to provide probability to identify whether its real fake
        model = tf.keras.Model(inp, out, name="discriminator_1d_ecg")
        return model

    generator = make_generator(latent_dim=latent_dim, out_len=timesteps)
    discriminator = make_discriminator(input_len=timesteps)
# Setting optimizers and parameters, learning rate of discriminator is higher because it learns faster and gives to generator
# beta is for momentum of the Adam optimizer, set values are typical values for GAN networks
    d_opt = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5, beta_2=0.9)
    g_opt = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)
# We use binary crossentropy as we are solving a two class problem (real or fake)
    discriminator.compile(
        optimizer=d_opt,
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    # Combined model for generator updates
    z_in = layers.Input(shape=(latent_dim,))
    fake_out = generator(z_in)
    discriminator.trainable = False  # freeze discriminator for generator updates
    validity = discriminator(fake_out)
    combined = tf.keras.Model(z_in, validity, name="gan_combined")
# the above freeze ensures that discriminator and generator work sequentially
# so that the generator receives meaningful gradients from the discriminator that are updated
# else they would go together and the generator wont benefit from newly updated values
    combined.compile(
        optimizer=g_opt,
        loss="binary_crossentropy",
    )

    # re-enable trainability for discriminator for when we call train_on_batch on it
    discriminator.trainable = True

    def smooth_labels(size, real=True):
        if real:
            # real labels in [0.8, 1.0]
            return np.random.uniform(0.8, 1.0, size=(size, 1)).astype(np.float32)
        else:
            # fake labels in [0.0, 0.2]
            return np.random.uniform(0.0, 0.2, size=(size, 1)).astype(np.float32)

    def add_instance_noise(x, sigma=0.05):
        noise = np.random.normal(0.0, sigma, size=x.shape).astype(np.float32)
        return (x + noise).astype(np.float32)

    print("Starting GAN training...")
    for step in range(1, steps + 1):
# Train discriminator
        for _ in range(d_updates_per_step):
            idx = np.random.randint(0, X_pos.shape[0], size=batch_size)
            real_batch = X_pos[idx].astype(np.float32)

            # instance noise on real
            real_batch = add_instance_noise(real_batch, sigma=0.05)

            z = np.random.normal(0, 1, (batch_size, latent_dim)).astype(np.float32)
            fake_batch = generator.predict(z, verbose=0)

            # instance noise on fake
            fake_batch = add_instance_noise(fake_batch, sigma=0.05)

            y_real = smooth_labels(batch_size, real=True)
            y_fake = smooth_labels(batch_size, real=False)

            # occasionally flip a small fraction of labels (label noise)
            if np.random.rand() < 0.05:
                flip_idx = np.random.choice(batch_size, size=batch_size // 10, replace=False)
                y_real[flip_idx], y_fake[flip_idx] = y_fake[flip_idx], y_real[flip_idx]

            discriminator.trainable = True
            d_loss_real = discriminator.train_on_batch(real_batch, y_real)
            d_loss_fake = discriminator.train_on_batch(fake_batch, y_fake)

            d_loss_real = np.array(d_loss_real)
            d_loss_fake = np.array(d_loss_fake)
            d_loss = 0.5 * (d_loss_real + d_loss_fake)
# Train generator:
        for _ in range(g_updates_per_step):
            z = np.random.normal(0, 1, (batch_size, latent_dim)).astype(np.float32)
            # generator wants D to output "real" (smoothed)
            y_gen = smooth_labels(batch_size, real=True)

            discriminator.trainable = False
            g_loss = combined.train_on_batch(z, y_gen)
# The following is for logging:
        if step % print_interval == 0 or step == 1:
            d_arr = np.array(d_loss).ravel()
            d_loss_val = float(d_arr[0])
            d_acc_val = float(d_arr[1]) if d_arr.size > 1 else float("nan")

            g_arr = np.array(g_loss).ravel()
            g_loss_val = float(g_arr[0])

            print(
                f"[GAN step {step}/{steps}] "
                f"D_loss: {d_loss_val:.4f}, D_acc: {d_acc_val:.3f} | "
                f"G_loss: {g_loss_val:.4f}"
            )


    print("GAN training finished.")


GAN training set (positives only): (1723, 2500, 1)
Starting GAN training...
[GAN step 1/6000] D_loss: 0.7302, D_acc: 0.000 | G_loss: 0.4716
[GAN step 200/6000] D_loss: 0.3526, D_acc: 0.000 | G_loss: 0.3521
[GAN step 400/6000] D_loss: 0.3441, D_acc: 0.000 | G_loss: 0.3415
[GAN step 600/6000] D_loss: 0.3531, D_acc: 0.000 | G_loss: 0.6153
[GAN step 800/6000] D_loss: 0.3514, D_acc: 0.000 | G_loss: 0.8782
[GAN step 1000/6000] D_loss: 0.3518, D_acc: 0.000 | G_loss: 1.0176
[GAN step 1200/6000] D_loss: 0.3534, D_acc: 0.000 | G_loss: 1.0681
[GAN step 1400/6000] D_loss: 0.3568, D_acc: 0.000 | G_loss: 1.1476
[GAN step 1600/6000] D_loss: 0.3627, D_acc: 0.000 | G_loss: 1.0962
[GAN step 1800/6000] D_loss: 0.3745, D_acc: 0.000 | G_loss: 1.2342
[GAN step 2000/6000] D_loss: 0.3743, D_acc: 0.000 | G_loss: 1.1694
[GAN step 2200/6000] D_loss: 0.3814, D_acc: 0.000 | G_loss: 1.1074
[GAN step 2400/6000] D_loss: 0.3806, D_acc: 0.000 | G_loss: 1.0727
[GAN step 2600/6000] D_loss: 0.3918, D_acc: 0.000 | G_loss: 

In [6]:
# The purpose of this cell is to evaluate the GAN model
# We first sample a batch of reals and fakes
idx = np.random.randint(0, X_pos.shape[0], size=512)
real_batch = X_pos[idx]
z = np.random.normal(0, 1, (512, latent_dim)).astype(np.float32)
fake_batch = generator.predict(z, verbose=0)
# then we check how good the discriminator is, real score should be high, fake should be low
real_scores = discriminator.predict(real_batch, verbose=0).ravel()
fake_scores = discriminator.predict(fake_batch, verbose=0).ravel()

print("Real score mean±sd:", real_scores.mean(), real_scores.std())
print("Fake score mean±sd:", fake_scores.mean(), fake_scores.std())


Real score mean±sd: 0.9153381 0.047951344
Fake score mean±sd: 0.16553685 0.08524102


In [7]:
# This cell is to use GAN to synthesize hypoglycemic signals
target_pos_ratio = 0.30

pos_mask = (y_train == 1)
neg_mask = (y_train == 0)
n_pos = int(pos_mask.sum())
n_neg = int(neg_mask.sum())

desired_pos = (target_pos_ratio / (1.0 - target_pos_ratio)) * n_neg
m_needed = int(max(0, math.floor(desired_pos - n_pos + 1e-9)))

# Safety cap to avoid flooding with low-diversity fakes
max_synth = max(0, 2 * n_pos)
m_needed = min(m_needed, max_synth)

print(f"Current train pos: {n_pos}, neg: {n_neg} → synth positives needed: {m_needed}")

def _per_segment_renorm(x):
    # normalize per-segment to match distribution of real ECG windows
    m = x.mean(axis=0, keepdims=True)
    s = x.std(axis=0, keepdims=True) + 1e-8
    return (x - m) / s

if m_needed > 0 and n_pos > 0:
    generated = 0
    kept_chunks = []

    while generated < m_needed:
        bs = min(batch_size, m_needed - generated)

        # Generate latent noise
        z = np.random.normal(0, 1, (bs, latent_dim)).astype(np.float32)

        # Generate fake ECG segments
        fake_batch = generator.predict(z, verbose=0).astype(np.float32)    # (bs, 2500, 1)

        # Optional: discriminator-based gating
        try:
            scores = discriminator.predict(fake_batch, verbose=0).reshape(-1)  # probability "real"
            keep_mask = scores > 0.5
            fake_batch = fake_batch[keep_mask]
        except Exception:
            # If D not ready, skip gating
            pass

        # If gating removed everything, fallback to the ungated batch
        if fake_batch.size == 0:
            fake_batch = generator.predict(z, verbose=0).astype(np.float32)

        # Use true batch size after gating
        n_batch = fake_batch.shape[0]
        seq_len = fake_batch.shape[1]

        # Renormalize
        fake_batch = _per_segment_renorm(fake_batch.reshape(n_batch, -1))
        fake_batch = fake_batch.reshape(n_batch, seq_len, 1)

        # Clip outliers for safety
        fake_batch = np.clip(fake_batch, -3.5, 3.5).astype(np.float32)

        # Only add up to the amount still needed
        kept = min(fake_batch.shape[0], m_needed - generated)
        kept_chunks.append(fake_batch[:kept])
        generated += kept

    # Final concatenation of all synthetic positives
    X_synth = np.concatenate(kept_chunks, axis=0)[:m_needed]     # (m_needed, 2500, 1)
    y_synth = np.ones((X_synth.shape[0],), dtype=int)

    # Augment training set
    X_train_aug = np.concatenate([X_train, X_synth], axis=0).astype(np.float32)
    y_train_aug = np.concatenate([y_train, y_synth], axis=0)
else:
    print("No synthetic positives added (either none needed or no positives available to train GAN).")
    X_train_aug = X_train.copy().astype(np.float32)
    y_train_aug = y_train.copy()

# Shuffle augmented training set
perm = np.random.permutation(X_train_aug.shape[0])
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

pos_ratio = float(y_train_aug.mean())
print("After GAN aug - train:", X_train_aug.shape, "pos ratio:", f"{pos_ratio:.3f}")


Current train pos: 1723, neg: 30218 → synth positives needed: 3446
After GAN aug - train: (35387, 2500, 1) pos ratio: 0.146


In [8]:


# Here we build our main classifier mode, 3 1d CNN layers for feature extraction followed with one BiLSTM layer
# With dropout to avoid overfitting andfinally one dense layer for the output with a sigmoid to give us a probability whether its hypoglycemic or not
def build_cnn_bilstm(timesteps, channels=1, lr=1e-3):
    inp = layers.Input(shape=(timesteps, channels))

    x = layers.Conv1D(32, kernel_size=7, strides=2, padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(64, kernel_size=7, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, kernel_size=7, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision'),
        ]
    )
    return model

clf = build_cnn_bilstm(timesteps, channels=1)
clf.summary()
# We define early stopping based on auc with a patience of 20 to avoid unessecary training
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=20, restore_best_weights=True
)
# if it early stops, we save the model that had the best score to use later

history = clf.fit(
    X_train_aug, y_train_aug,
    epochs=100,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=2
)



Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 2500, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 1250, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1250, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 1250, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 625, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 625, 64)        │        14,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 625, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 625, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 312, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 312, 128)       │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 312, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 312, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 156, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 156, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 180,161 (703.75 KB)

 Trainable params: 179,713 (702.00 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/100
277/277 - 22s - 81ms/step - auc: 0.9218 - loss: 0.1974 - precision: 0.9166 - recall: 0.6127 - val_auc: 0.5760 - val_loss: 0.3216 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/100
277/277 - 12s - 43ms/step - auc: 0.9461 - loss: 0.1499 - precision: 0.9549 - recall: 0.7168 - val_auc: 0.6778 - val_loss: 0.2932 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
277/277 - 12s - 44ms/step - auc: 0.9535 - loss: 0.1385 - precision: 0.9580 - recall: 0.7421 - val_auc: 0.8456 - val_loss: 0.2286 - val_precision: 0.1935 - val_recall: 0.1138
Epoch 4/100
277/277 - 12s - 43ms/step - auc: 0.9605 - loss: 0.1281 - precision: 0.9566 - recall: 0.7593 - val_auc: 0.8841 - val_loss: 0.1589 - val_precision: 0.5000 - val_recall: 0.0352
Epoch 5/100
277/277 - 12s - 42ms/step - auc: 0.9680 - loss: 0.1191 - precision: 0.9600 - recall: 0.7657 - val_auc: 0.9018 - val_loss: 0.1765 - val_precision: 0.4453 - val_recall: 0.4526
Epoch 6/100
277/277 - 12s - 42ms/step - auc: 0.9736 - 

In [9]:

# We evaluate the best threshold to use by testing different threshold values on the validation set
# based on the value we get, we do the testing on the actual test set
val_probs = clf.predict(X_val).ravel()
best_th = 0.5
best_f1 = -1.0
best_prec = 0.0
best_rec  = 0.0

for th in np.linspace(0.05, 0.95, 19):
    y_pred_val = (val_probs >= th).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_val, y_pred_val, average='binary', zero_division=0
    )
    if f1 > best_f1:
        best_f1, best_th, best_prec, best_rec = f1, th, prec, rec

print(f"Best threshold on val: {best_th:.3f}")
print(f"Val F1: {best_f1:.3f}, Recall: {best_rec:.3f}, Precision: {best_prec:.3f}")


214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
Best threshold on val: 0.850
Val F1: 0.747, Recall: 0.678, Precision: 0.833


In [20]:

# Although the best_threshold on validation was 0.85, through trial we discovered that 0.8 gives an even better result
test_probs = clf.predict(X_test).ravel()
y_pred_test = (test_probs >= 0.8).astype(int)

prec_t, rec_t, f1_t, _ = precision_recall_fscore_support(
    y_test, y_pred_test, average='binary', zero_division=0
)
acc_t = (y_pred_test == y_test).mean()
auc_t = roc_auc_score(y_test, test_probs)
cm_t  = confusion_matrix(y_test, y_pred_test)
# Print out all the metrics and matrix
print("=== Test metrics (random split, NO patient-wise separation) ===")
print(f"Accuracy: {acc_t:.3f}")
print(f"Recall (hypo): {rec_t:.3f}")
print(f"Precision (hypo): {prec_t:.3f}")
print(f"F1 (hypo): {f1_t:.3f}")
print(f"AUROC: {auc_t:.3f}")
print("\nConfusion matrix (rows=true, cols=pred):\n", cm_t)


214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
=== Test metrics (random split, NO patient-wise separation) ===
Accuracy: 0.977
Recall (hypo): 0.759
Precision (hypo): 0.805
F1 (hypo): 0.781
AUROC: 0.981

Confusion matrix (rows=true, cols=pred):
 [[6408   68]
 [  89  280]]
